In [2]:
! pip install pmdarima

In [3]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb

from statsmodels.tsa.arima.model import ARIMA
import pmdarima as pm

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

import joblib
import pickle
import os

# Set random seeds
np.random.seed(42)
tf.random.set_seed(42)

In [4]:
# Load and prepare data
df = pd.read_csv('processed_food_demand_dataset.csv')
df = df.drop(columns=['id'])  # Remove unnecessary column

target = 'num_orders'
feature_columns = [col for col in df.columns if col != target]

X = df[feature_columns].copy()
y = df[target].copy()

print(f"Dataset shape: {df.shape}")
print(f"Features: {len(feature_columns)}")

Dataset shape: (456548, 32)
Features: 31


In [5]:
# Encode categorical variables
categorical_cols = ['center_id', 'meal_id', 'city_code', 'region_code']
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le

X = X.fillna(X.median())

In [6]:
# Train-test split (chronological)
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [7]:
# Evaluation function
def evaluate_model(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-10))) * 100
    return {'Model': name, 'MAE': round(mae, 2), 'RMSE': round(rmse, 2), 
            'R2': round(r2, 4), 'MAPE': round(mape, 2)}

In [8]:
# Train models
print("Training models...")

# Linear Regression
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)
lr_metrics = evaluate_model(y_test, y_pred_lr, "Linear Regression")

# XGBoost
xgb_model = xgb.XGBRegressor(n_estimators=200, max_depth=10, learning_rate=0.1, 
                              random_state=42, n_jobs=-1)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)
xgb_metrics = evaluate_model(y_test, y_pred_xgb, "XGBoost")

Training models...


In [9]:
# ARIMA (weekly aggregated)
weekly_demand = df.groupby('week')['num_orders'].sum().reset_index()
auto_model = pm.auto_arima(weekly_demand['num_orders'], start_p=0, max_p=3, 
                           start_q=0, max_q=3, seasonal=False, stepwise=True, 
                           trace=False, error_action='ignore', suppress_warnings=True)
arima = ARIMA(weekly_demand['num_orders'], order=auto_model.order)
arima_fitted = arima.fit()

In [10]:
# LSTM (simplified)
def create_sequences(data, seq_length=5):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length])
    return np.array(X), np.array(y)

seq_length = 5
X_lstm, y_lstm = create_sequences(weekly_demand['num_orders'].values, seq_length)
split = int(len(X_lstm) * 0.8)

X_train_lstm = X_lstm[:split].reshape(-1, seq_length, 1)
X_test_lstm = X_lstm[split:].reshape(-1, seq_length, 1)
y_train_lstm, y_test_lstm = y_lstm[:split], y_lstm[split:]

lstm = Sequential([
    LSTM(50, activation='relu', return_sequences=True, input_shape=(seq_length, 1)),
    Dropout(0.2),
    LSTM(50, activation='relu'),
    Dropout(0.2),
    Dense(1)
])
lstm.compile(optimizer=Adam(0.001), loss='mse')
lstm.fit(X_train_lstm, y_train_lstm, validation_data=(X_test_lstm, y_test_lstm),
         epochs=50, batch_size=8, callbacks=[EarlyStopping(patience=5)], verbose=0)

In [11]:
# Compare and select best model
comparison = pd.DataFrame([lr_metrics, xgb_metrics])
best_idx = comparison['R2'].idxmax()
best_model_name = comparison.loc[best_idx, 'Model']

print("\nModel Comparison:")
print(comparison.to_string(index=False))
print(f"\nBest Model: {best_model_name}")


Model Comparison:
            Model    MAE   RMSE     R2   MAPE
Linear Regression 158.15 279.47 0.4170 156.72
          XGBoost  82.88 178.06 0.7633  59.84

Best Model: XGBoost


In [12]:
# Save all models and objects
os.makedirs('models', exist_ok=True)
joblib.dump(lr, 'models/linear_regression.pkl')
joblib.dump(xgb_model, 'models/xgboost.pkl')
joblib.dump(arima_fitted, 'models/arima.pkl')
lstm.save('models/lstm_model.h5')
joblib.dump(scaler, 'models/scaler.pkl')
joblib.dump(label_encoders, 'models/label_encoders.pkl')
joblib.dump(feature_columns, 'models/feature_columns.pkl')

print("\nAll models saved to 'models' folder")


All models saved to 'models' folder


In [13]:
# PREDICTION FUNCTIONS
def predict_demand(features, model_name='xgboost'):
    """
    Predict number of orders based on input features
    
    Parameters:
    features: dict or DataFrame with all required features
    model_name: 'linear_regression' or 'xgboost' (default)
    
    Returns:
    predicted_orders: int
    """
    try:
        # Convert dict to DataFrame if needed
        if isinstance(features, dict):
            features = pd.DataFrame([features])
        
        # Load model
        if model_name == 'linear_regression':
            model = joblib.load('models/linear_regression.pkl')
            scaler = joblib.load('models/scaler.pkl')
            features_scaled = scaler.transform(features)
            pred = model.predict(features_scaled)[0]
        else:  # xgboost (default)
            model = joblib.load('models/xgboost.pkl')
            pred = model.predict(features)[0]
        
        return int(round(pred))
    
    except Exception as e:
        print(f"Prediction error: {e}")
        return None


def predict_week(last_row_features, model_name='xgboost'):
    """
    Predict demand for next 7 days
    """
    predictions = []
    current = last_row_features.copy()
    
    for day in range(7):
        if isinstance(current, dict):
            current_df = pd.DataFrame([current])
        else:
            current_df = current.copy()
        
        if 'week' in current_df.columns:
            current_df['week'] = current_df['week'] + 1
        
        pred = predict_demand(current_df, model_name)
        predictions.append(pred)
        
        # Update for next iteration
        if isinstance(current, dict):
            current['week'] = current.get('week', 0) + 1
    
    return predictions

In [30]:
# TEST THE FUNCTIONS

# Test with sample from dataset
sample_input = X_test.iloc[0:1].copy()
actual = y_test.iloc[0]

print("Single Prediction Test:")
print(f"Actual: {actual}")
for model in ['linear_regression', 'xgboost']:
    pred = predict_demand(sample_input, model)
    print(f"{model}: {pred}")

print("\n" + "="*50)

# Test with custom input
sample_custom = {
    'week': 200,
    'center_id': 1,
    'meal_id': 101,
    'checkout_price': 189,
    'base_price': 185,
    'emailer_for_promotion': 0,
    'homepage_featured': 1,
    'city_code': 63100,
    'region_code': 300,
    'op_area': 125,

    'category_Beverages': 9,
    'category_Biryani': 0,
    'category_Desert': 0,
    'category_Extras': 0,
    'category_Fish': 0,
    'category_Other Snacks': 0,
    'category_Pasta': 0,
    'category_Pizza': 1,
    'category_Rice Bowl': 0,
    'category_Salad': 0,
    'category_Sandwich': 0,
    'category_Seafood': 0,
    'category_Soup': 0,
    'category_Starters': 0,

    'cuisine_Continental': 0,
    'cuisine_Indian': 0,
    'cuisine_Italian': 1,
    'cuisine_Thai': 0,

    'center_type_TYPE_A': 1,
    'center_type_TYPE_B': 0,
    'center_type_TYPE_C': 0
}
print("Custom Input Prediction:")
pred = predict_demand(sample_custom, 'xgboost')
print(f"Predicted orders: {pred}")

Single Prediction Test:
Actual: 163
linear_regression: 383
xgboost: 262

Custom Input Prediction:
Predicted orders: 542


In [32]:
models = {
    'linear_regression': lr,   # your trained LR model
    'xgboost': xgb_model                        # your trained XGB model
}

# ================== COMPLETE TESTING CELL ==================

from sklearn.metrics import mean_absolute_error, r2_score
import numpy as np

print("========== SINGLE PREDICTION TEST ==========")

sample_input = X_test.iloc[0:1].copy()
actual = y_test.iloc[0]

print(f"Actual Value: {actual}")

for model_name in ['linear_regression', 'xgboost']:
    pred = predict_demand(sample_input, model_name)
    
    error = abs(actual - pred)
    error_percent = (error / actual) * 100

    print(f"\nModel: {model_name}")
    print(f"Predicted: {pred}")
    print(f"Error: {error}")
    print(f"Error %: {error_percent:.2f}%")

print("\n" + "="*60)


# ================== FULL MODEL PERFORMANCE ==================

print("\n========== OVERALL MODEL PERFORMANCE ==========")

for model_name, model in models.items():   # models = {'linear_regression': model1, 'xgboost': model2}
    
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(np.mean((y_test - y_pred) ** 2))
    r2 = r2_score(y_test, y_pred)

    print(f"\nModel: {model_name}")
    print(f"MAE: {mae}")
    print(f"RMSE: {rmse}")
    print(f"R2 Score: {r2}")
    print(f"Accuracy: {r2 * 100:.2f}%")

print("\n" + "="*60)


# ================== MULTIPLE SAMPLE TEST ==================

print("\n========== MULTIPLE SAMPLE TEST ==========")

sample_data = X_test.iloc[:10]

for model_name, model in models.items():
    preds = model.predict(sample_data)

    print(f"\nModel: {model_name}")
    
    for i in range(len(sample_data)):
        actual_val = y_test.iloc[i]
        pred_val = preds[i]
        error = abs(actual_val - pred_val)

        print(f"Actual: {actual_val}, Predicted: {pred_val}, Error: {error}")

print("\n" + "="*60)


# ================== CUSTOM INPUT TEST ==================

sample_custom = {
    'week': 200,
    'center_id': 1,
    'meal_id': 101,
    'checkout_price': 189,
    'base_price': 185,
    'emailer_for_promotion': 0,
    'homepage_featured': 1,
    'city_code': 63100,
    'region_code': 300,
    'op_area': 125,

    'category_Beverages': 9,
    'category_Biryani': 0,
    'category_Desert': 0,
    'category_Extras': 0,
    'category_Fish': 0,
    'category_Other Snacks': 0,
    'category_Pasta': 0,
    'category_Pizza': 1,
    'category_Rice Bowl': 0,
    'category_Salad': 0,
    'category_Sandwich': 0,
    'category_Seafood': 0,
    'category_Soup': 0,
    'category_Starters': 0,

    'cuisine_Continental': 0,
    'cuisine_Indian': 0,
    'cuisine_Italian': 1,
    'cuisine_Thai': 0,

    'center_type_TYPE_A': 1,
    'center_type_TYPE_B': 0,
    'center_type_TYPE_C': 0
}

print("\n========== CUSTOM INPUT PREDICTION ==========")

pred = predict_demand(sample_custom, 'xgboost')
print(f"Predicted orders: {pred}")

# NOTE: No accuracy here because actual value is unknown

========== SINGLE PREDICTION TEST ==========
Actual Value: 163

Model: linear_regression
Predicted: 383
Error: 220
Error %: 134.97%

Model: xgboost
Predicted: 262
Error: 99
Error %: 60.74%


========== OVERALL MODEL PERFORMANCE ==========

Model: linear_regression
MAE: 1010.8245010241591
RMSE: 1198.0150656551884
R2 Score: -9.712836802179728
Accuracy: -971.28%

Model: xgboost
MAE: 82.88117218017578
RMSE: 178.06486602113682
R2 Score: 0.7633337378501892
Accuracy: 76.33%


========== MULTIPLE SAMPLE TEST ==========

Model: linear_regression
Actual: 163, Predicted: -1261.6648933733209, Error: 1424.6648933733209
Actual: 1592, Predicted: -1132.790163821782, Error: 2724.790163821782
Actual: 878, Predicted: 36.05168777734565, Error: 841.9483122226543
Actual: 851, Predicted: 145.9946790851791, Error: 705.0053209148209
Actual: 148, Predicted: 166.80504923764698, Error: 18.805049237646983
Actual: 80, Predicted: 10.05647462277895, Error: 69.94352537722105
Actual: 149, Predicted: -276.3544479732498,

In [37]:
import random

print("========== RANDOM TEST FROM DATASET ==========")

# pick random row
random_index = random.randint(0, len(X_test)-1)

sample_input = X_test.iloc[random_index:random_index+1]
actual = y_test.iloc[random_index]

print(f"Random Index: {random_index}")
print(f"Actual Value: {actual}")

for model_name in ['linear_regression', 'xgboost']:
    pred = predict_demand(sample_input, model_name)
    
    error = abs(actual - pred)
    error_percent = (error / actual) * 100

    print(f"\nModel: {model_name}")
    print(f"Predicted: {pred}")
    print(f"Error: {error}")
    print(f"Error %: {error_percent:.2f}%")

========== RANDOM TEST FROM DATASET ==========
Random Index: 46899
Actual Value: 68

Model: linear_regression
Predicted: 289
Error: 221
Error %: 325.00%

Model: xgboost
Predicted: 87
Error: 19
Error %: 27.94%


In [31]:
print("========== MANUAL INPUT TEST ==========")

# 👉 Step 1: Enter your own input here
custom_input = {
    'week': 200,
    'center_id': 1,
    'meal_id': 101,
    'checkout_price': 189,
    'base_price': 185,
    'emailer_for_promotion': 0,
    'homepage_featured': 1,
    'city_code': 63100,
    'region_code': 300,
    'op_area': 125,

    'category_Beverages': 9,
    'category_Biryani': 0,
    'category_Desert': 0,
    'category_Extras': 0,
    'category_Fish': 0,
    'category_Other Snacks': 0,
    'category_Pasta': 0,
    'category_Pizza': 1,
    'category_Rice Bowl': 0,
    'category_Salad': 0,
    'category_Sandwich': 0,
    'category_Seafood': 0,
    'category_Soup': 0,
    'category_Starters': 0,

    'cuisine_Continental': 0,
    'cuisine_Indian': 0,
    'cuisine_Italian': 1,
    'cuisine_Thai': 0,

    'center_type_TYPE_A': 1,
    'center_type_TYPE_B': 0,
    'center_type_TYPE_C': 0
}

# 👉 Step 2: (OPTIONAL) Enter actual value if you know it
actual_value = 163

# 👉 Step 3: Run prediction
for model_name in ['linear_regression', 'xgboost']:
    
    pred = predict_demand(custom_input, model_name)

    print(f"\nModel: {model_name}")
    print(f"Predicted: {pred}")

    # 👉 Step 4: Check error ONLY if actual value exists
    if actual_value is not None:
        error = abs(actual_value - pred)
        error_percent = (error / actual_value) * 100

        print(f"Actual: {actual_value}")
        print(f"Error: {error}")
        print(f"Error %: {error_percent:.2f}%")

========== MANUAL INPUT TEST ==========

Model: linear_regression
Predicted: -339
Actual: 163
Error: 502
Error %: 307.98%

Model: xgboost
Predicted: 542
Actual: 163
Error: 379
Error %: 232.52%


In [ ]:
print("\n" + "="*60)
print("MODEL TRAINING COMPLETED")
print("="*60)
print(f"\nBest model: {best_model_name}")
print(f"R2 score: {comparison.loc[best_idx, 'R2']}")
print("\nUse predict_demand() for single week predictions")

In [ ]:
raw_materials_per_meal = {

    'Beverages': {
        'Water': 250,
        'Sugar': 20,
        'Flavor/Syrup': 10
    },

    'Rice Bowl': {
        'Rice': 200,
        'Vegetables': 100,
        'Sauce': 30
    },

    'Starters': {
        'Chicken': 120,
        'Oil': 30,
        'Spices': 10
    },

    'Pasta': {
        'Pasta': 150,
        'Sauce': 80,
        'Cheese': 40
    },

    'Sandwich': {
        'Bread': 2,
        'Chicken/Vegetables': 100,
        'Sauce': 20
    },

    'Biryani': {
        'Rice': 250,
        'Chicken': 150,
        'Spices': 20
    },

    'Pizza': {
        'Flour': 120,
        'Cheese': 80,
        'Sauce': 40
    },

    'Seafood': {
        'Fish/Shrimp': 180,
        'Oil': 40,
        'Spices': 15
    },

    'Desert': {
        'Milk': 200,
        'Sugar': 50,
        'Cream': 40
    },

    'Soup': {
        'Vegetables': 120,
        'Water/Stock': 250,
        'Spices': 10
    },

    'Extras': {
        'Sauce': 20,
        'Bread': 1
    },

    'Other Snacks': {
        'Flour': 100,
        'Oil': 50,
        'Spices': 10
    }
}

In [ ]:
def predict_orders(input_data, model):
    predictions = model.predict(input_data)
    return predictions

def calculate_food_and_inventory(input_data, predictions, category_cols):
    
    for idx in range(len(input_data)):
        row = input_data.iloc[idx]

        # Identify meal category
        categories = [c.replace('category_', '') for c in category_cols if row[c] == 1]

        if categories:
            main_cat = categories[0]

            print("Meal Category:", main_cat)
            print("Predicted Orders:", int(predictions[idx]))

            # Food demand
            food_demand = int(predictions[idx])

            if main_cat in raw_materials_per_meal:
                print("Required Raw Materials:")

                for material, qty in raw_materials_per_meal[main_cat].items():
                    required = qty * food_demand
                    print(f"{material}: {required} grams")

        print("---------------------------")

In [ ]:
category_cols = [col for col in X_test.columns if col.startswith('category_')]

# Take sample input
next_day_input = X_test.iloc[:5]

# Predict orders
pred_orders = predict_orders(next_day_input, xgb_model)

# Calculate food demand and inventory
calculate_food_and_inventory(next_day_input, pred_orders, category_cols)